# Eco-Travel Advisor — Setup, Testing & Demo

Conversational agent for sustainable tourism planning, built on **Rasa Open Source**.
This notebook is the single reproducible entry point for the project: it provisions the
knowledge base, validates and trains the assistant, evaluates it, and serves the REST
backend for the web UI.

**Two Python environments side by side.** The data layer (this kernel) runs on Python 3.12,
while Rasa 3.6.x requires Python 3.10, so Part B builds an isolated 3.10 virtual environment
and calls Rasa through its own binary. The two environments do not interfere with each other.

**Structure**
- **Part A — Data & knowledge base** (Python 3.12 kernel): secrets, seed validation, NeonDB,
  the fallback cascade, and the live external-API integration checks.
- **Part B — Conversational assistant** (Python 3.10 venv): Rasa install, validation, training
  and evaluation.
- **Part C — Serve, integrate & analyse**: run the backend, exercise the REST contract, expose
  a public URL, and inspect a conversation tracker.

**Security.** Every secret (the database URL and all API keys) is read from the environment
only — through the Colab *Secrets* panel or a local `.env` file. Values are never printed,
logged or committed; cells display only a boolean *configured?* flag.

# Part A — Data & knowledge base

## 1. Environment detection

Detects whether the notebook runs on Google Colab or a local Jupyter server, so later cells can branch on it.

In [ ]:
import os, sys

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print('Environment:', 'Google Colab' if IN_COLAB else 'local Jupyter')

## 2. Prepare the project

On **Colab** this always pulls a *fresh* clone from GitHub, so every run reflects the latest
push and no stale files survive between sessions. On **local Jupyter** the notebook already sits
at the project root, so it simply resolves the path.

In [ ]:
REPO_URL = 'https://github.com/yasmiiinay/eco-travel-advisor.git'
PROJECT_DIR = 'eco-travel-advisor'

if IN_COLAB:
    # Always fetch a fresh copy so every run reflects the latest GitHub push.
    if os.path.basename(os.getcwd()) == PROJECT_DIR:
        os.chdir('..')                       # step out of a previous clone
    !rm -rf $PROJECT_DIR
    !git clone $REPO_URL
    if not os.path.isdir(PROJECT_DIR):
        raise FileNotFoundError('git clone failed — check REPO_URL and that the repo is public.')
    os.chdir(PROJECT_DIR)
    PROJECT_ROOT = os.getcwd()
else:
    # Local Jupyter: the notebook already lives at the project root.
    if not (os.path.isdir('actions') and os.path.isdir('data/seed')):
        raise FileNotFoundError('Open this notebook from the project root '
                                '(the folder containing actions/ and data/seed/).')
    PROJECT_ROOT = os.getcwd()

print('Project root:', PROJECT_ROOT)

## 3. Install dependencies

Installs the lightweight data-layer libraries used by this kernel (the heavy Rasa stack is installed separately in Part B).

In [ ]:
%pip install -q sqlalchemy psycopg2-binary python-dotenv requests

## 4. Load secrets safely

- **Colab:** add the secrets in the left-hand **Secrets** panel and enable notebook access.
- **Local:** put them in a `.env` file at the project root (never committed).

| Secret | Purpose | Required? |
|---|---|---|
| `NEON_DATABASE_URL` | Serverless PostgreSQL knowledge base | optional (JSON fallback otherwise) |
| `CLIMATIQ_API_KEY` | Live carbon-emission factors | optional |
| `AVIATIONSTACK_API_KEY` | Live flight sample for the transport card | optional |
| `OPENCAGE_API_KEY` | Reverse-geocode GPS to a friendly place name | optional |
| `OPENROUTESERVICE_API_KEY` | Real road-routed distance for ground transport | optional |

The assistant runs fully without any of these (it uses the curated seed data and stored
factors). Only a boolean *configured?* flag is printed — the actual values are never shown.

In [ ]:
SECRET_KEYS = (
    'NEON_DATABASE_URL',
    'CLIMATIQ_API_KEY',
    'AVIATIONSTACK_API_KEY',
    'OPENCAGE_API_KEY',
    'OPENROUTESERVICE_API_KEY',
)

def load_secrets():
    # Load secrets into the environment from Colab Secrets or a local .env file.
    if IN_COLAB:
        from google.colab import userdata
        for key in SECRET_KEYS:
            try:
                value = userdata.get(key)
                if value:
                    os.environ[key] = value
            except Exception:
                pass  # secret not set / access not granted — stays unconfigured
    else:
        try:
            from dotenv import load_dotenv
            load_dotenv()
        except ImportError:
            print('python-dotenv not installed; relying on shell environment.')

load_secrets()
for key in SECRET_KEYS:
    print(f'{key:24} configured: {bool(os.environ.get(key))}')

## 5. Validate the curated seed data

Checks that every seed file is valid JSON and internally consistent: foreign keys resolve,
sustainability tags exist, every transport mode has an emission factor, and the stored
emissions match `distance x factor`. This guards the knowledge base before it reaches the database.

In [ ]:
import json, glob

seed = {os.path.basename(p): json.load(open(p, encoding='utf-8'))
        for p in glob.glob('data/seed/*.json')}
print('Seed files loaded:', len(seed))

dest_ids = {d['destination_id'] for d in seed['destination.json']}
tag_names = {t['tag_name'] for t in seed['tags.json']}
factor_modes = {e['mode'] for e in seed['emission_factor.json']}
factor = {e['mode']: e['kg_co2e_per_passenger_km'] for e in seed['emission_factor.json']}
errors = []

for fn in ('hotel.json', 'experience.json', 'transport_option.json', 'offset_option.json'):
    for row in seed[fn]:
        if row['destination_id'] not in dest_ids:
            errors.append(f"{fn}: bad destination_id {row['destination_id']}")
        for tg in row.get('sustainability_tags', []):
            if tg not in tag_names:
                errors.append(f"{fn}: unknown tag '{tg}'")

for row in seed['transport_option.json']:
    if row['mode'] not in factor_modes:
        errors.append(f"transport_option.json: mode '{row['mode']}' has no emission factor")
    expected = round(row['estimated_distance_km'] * factor[row['mode']], 1)
    if abs(expected - row['estimated_emissions_kg_per_person']) > 0.2:
        errors.append(f"option {row['option_id']}: emissions mismatch")

for d in sorted(dest_ids):
    h = sum(1 for x in seed['hotel.json'] if x['destination_id'] == d)
    e = sum(1 for x in seed['experience.json'] if x['destination_id'] == d)
    print(f'  destination {d}: {h} hotels, {e} experiences')

print('\nVALIDATION:', 'PASSED' if not errors else f'{len(errors)} ERROR(S): {errors}')

## 6. Test the NeonDB connection

Imports the ORM models from `actions/db.py` and runs a trivial `SELECT 1`. Connection details
are never displayed. If the database is unreachable the project still works on the local JSON
fallback (demonstrated in section 9).

In [ ]:
sys.path.insert(0, os.path.join(PROJECT_ROOT, 'actions'))
import db

print('DB configured:', db.is_db_configured())
if db.is_db_configured():
    from sqlalchemy import text
    try:
        with db.get_engine().connect() as conn:
            conn.execute(text('SELECT 1'))
        print('NeonDB connection: OK')
    except Exception as exc:
        print('NeonDB connection FAILED:', type(exc).__name__)
else:
    print('NEON_DATABASE_URL not set — skipping (JSON fallback will be used).')

## 7. Seed the database

Runs `actions/seed_db.py`, which creates the tables and idempotently upserts every seed file. On the **first** run every table reports `inserted`.

In [ ]:
import subprocess

def run_seed():
    result = subprocess.run([sys.executable, 'actions/seed_db.py'],
                            capture_output=True, text=True)
    print(result.stdout)
    if result.returncode != 0:
        print('--- stderr ---')
        print(result.stderr)
    return result.returncode

run_seed()

## 8. Demonstrate idempotency

Re-running the seeder must **not** create duplicates: every table should now report `updated`
with `inserted = 0`. This is useful evidence for the testing section of the report.

In [ ]:
run_seed()

## 9. Resilience test — NeonDB to JSON fallback

Demonstrates the cascade **NeonDB -> local JSON**. The same query is run twice: first with the
database configured (`source: neondb`), then with the database forced off to prove the assistant
degrades gracefully to the local JSON seed files (`source: json_fallback`). This is direct
evidence for the resilience requirement.

In [ ]:
import importlib
import repository as repo

# --- Tier 1: NeonDB (the connection string was loaded in the secrets cell) ---
print('NEON_DATABASE_URL configured:', bool(os.environ.get('NEON_DATABASE_URL')))
dest, src1 = repo.resolve_destination('Pariiis')          # typo-tolerant
print(f"resolve_destination('Pariiis') -> {dest['city']}  | source: {src1}")
opts, src1b = repo.get_transport_options('London', dest['destination_id'])
print(f"greenest transport: {opts[0]['mode']} ~{opts[0]['estimated_emissions_kg_per_person']} kg  | source: {src1b}")

# --- Tier 2: force NeonDB OFF to prove graceful fallback to local JSON ---
saved = os.environ.pop('NEON_DATABASE_URL', None)
repo._seed_cache.clear()
dest2, src2 = repo.resolve_destination('Berln')           # another typo
print(f"\n[DB forced OFF] resolve_destination('Berln') -> {dest2['city']}  | source: {src2}")
opts2, src2b = repo.get_transport_options('Madrid', dest2['destination_id'])
print(f"[DB forced OFF] transport options for Madrid->{dest2['city']}: {len(opts2)} found  | source: {src2b}")
if saved:
    os.environ['NEON_DATABASE_URL'] = saved               # restore for later cells

print('\nFallback cascade verified:', src1 == 'neondb' and src2 == 'json_fallback')

## 10. Live external-API integration check

The assistant enriches its answers with four optional live services, each wrapped so a missing
key, timeout, rate-limit or bad response degrades quietly to the stored data. This section
verifies each one and the end-to-end chain that feeds the carbon estimate.

| Service | Used for | Provenance shown |
|---|---|---|
| **Climatiq** | Live emission factors | `data_source = climatiq` (else `neondb` / `json_fallback`) |
| **Aviationstack** | A real sample flight for the route | live line on the transport card (else omitted) |
| **OpenCage** | Friendly place name for a GPS fix | reverse-geocoded label (else nearest city only) |
| **OpenRouteService** | Road-routed distance for ground transport | `road-routed` distance (else great-circle) |

Carbon cascade: **Climatiq API -> stored factor (NeonDB -> local JSON) -> unavailable**. All keys
are read from the environment only and are never printed.

In [ ]:
import carbon, aviation, geo, routing
for m in (carbon, aviation, geo, routing):
    importlib.reload(m)

print('Configured:  climatiq=%s  aviationstack=%s  opencage=%s  ors=%s' % (
    carbon.is_climatiq_configured(), aviation.is_configured(),
    geo.is_configured(), routing.is_configured()))
print('-' * 64)

# (a) Climatiq — emission factor provenance across a few modes.
seen = set()
for mode, dist, pax in [('train', 956, 2), ('flight', 956, 1), ('coach', 344, 3)]:
    r = carbon.estimate_emissions(mode, dist, pax)
    seen.add(r['data_source'])
    print(f"[Climatiq] {mode:7} {dist} km x{pax}: {r['estimated_co2_kg']} kg | source={r['data_source']}")
print(f"           data_source values seen: {sorted(seen)}")

# (b) Aviationstack — a real sample flight for the route (None = no key / no live flight).
print(f"\n[Aviationstack] Paris -> Copenhagen sample flight: {aviation.get_flight_sample('Paris', 'Copenhagen')}")

# (c) OpenCage — GPS coordinates to a friendly place name.
city, name, dist = geo.resolve_location(50.11, 8.68)     # near Frankfurt
print(f"\n[OpenCage] nearest supported city={city} | friendly name={name} | {dist} km away")

# (d) OpenRouteService — real road distance between two supported cities.
o, d = geo.city_coords('Paris'), geo.city_coords('Amsterdam')
km = routing.routed_distance_km(o[0], o[1], d[0], d[1])
print(f"[OpenRouteService] Paris -> Amsterdam road distance: {km} km  (None = great-circle fallback)")

# (e) End-to-end: GPS -> nearest city -> ORS distance -> Climatiq carbon estimate.
end = carbon.estimate_emissions('train', km or 500, 1)
print(f"\n[End-to-end] routed {km} km fed to carbon estimate -> "
      f"{end['estimated_co2_kg']} kg | source={end['data_source']}")

# Part B — Conversational assistant (Rasa)

## 11. Rasa setup (isolated Python 3.10 environment)

Rasa 3.6.x needs **Python 3.10**, while this Colab kernel is 3.12, so we build a separate 3.10
virtual environment just for Rasa and call it through its own binary. The notebook kernel stays
3.12 (the data cells above keep working); the two environments sit side by side.

The first install pulls in TensorFlow and other heavy dependencies, so it takes a few minutes.

In [ ]:
# Paths to the Rasa environment (Colab) or the active local 3.10 venv.
RASA  = '/content/rasa-venv/bin/rasa'   if IN_COLAB else 'rasa'
PYBIN = '/content/rasa-venv/bin/python' if IN_COLAB else 'python'
print('Using rasa at:', RASA)

In [ ]:
# One-time: create the Python 3.10 venv and install the pinned requirements.
if IN_COLAB:
    !apt-get -qq install -y python3.10-venv python3.10-distutils >/dev/null
    !python3.10 -m venv /content/rasa-venv
    !/content/rasa-venv/bin/pip install -q --upgrade pip
    !/content/rasa-venv/bin/pip install -q -r requirements.txt
    !/content/rasa-venv/bin/rasa --version
else:
    print('Local: activate a Python 3.10 venv, then run  pip install -r requirements.txt')

## 12. Validate the project (imports + data consistency)

First confirm the custom actions import cleanly (rasa_sdk + repository + carbon), then run Rasa's
own validator over the domain, NLU, rules and stories.

In [ ]:
# Custom actions import test (no database connection is opened).
!{PYBIN} -c "import actions.actions; print('actions.py imports OK')"

In [ ]:
# Validate domain / nlu / rules / stories for consistency.
!{RASA} data validate

## 13. Train the assistant

Trains the NLU model and the dialogue policies defined in `config.yml`. The trained model is written to `models/` (git-ignored). This takes a couple of minutes on CPU.

In [ ]:
!{RASA} train

## 14. Evaluate the model (in-sample)

Evaluates the trained model on the training data — an optimistic, in-sample baseline. Reports
and a confusion matrix are written to `results/` (useful figures for the testing section). The
held-out figures in section 15 give the honest picture.

In [ ]:
!MPLBACKEND=Agg {RASA} test nlu -u data/nlu.yml
!MPLBACKEND=Agg {RASA} test core --stories data/stories.yml
!ls -R results 2>/dev/null || echo 'Results are written to the results/ folder.'

## 15. Robustness: held-out tests

Section 14 tests on the *training* data (optimistic). The cell below gives the honest, held-out
picture used in the report: unseen NLU examples (`tests/test_nlu_samples.yml`) and new dialogue
conversations (`tests/test_stories.yml`). All reports land in `results/`.

In [ ]:
# Held-out tests (~3 min): unseen NLU examples + new dialogue stories.
!MPLBACKEND=Agg {RASA} test nlu --nlu tests/test_nlu_samples.yml
!MPLBACKEND=Agg {RASA} test core --stories tests/test_stories.yml
!ls results

### Optional: 5-fold cross-validation (slow, run once for the report)

This retrains the NLU model on 5 folds, so it takes roughly 10-12 minutes. You only need it once
to obtain the honest held-out NLU F1 for the report — **skip it on routine runs**. (Reduce to
`--folds 3` for a faster, still-valid estimate.)

In [ ]:
#!MPLBACKEND=Agg {RASA} test nlu --nlu data/nlu.yml --cross-validation --folds 5

# Part C — Serve, integrate & analyse

## 16. Try the assistant locally (interactive shell)

`rasa shell` is interactive, so it is best run in a local terminal:

```
rasa run actions     # terminal 1 — custom actions, port 5055
rasa shell           # terminal 2 — chat with the bot
```

For the responsive web UI in `frontend/`, run the REST backend below (section 17) or use the
HuggingFace Spaces deployment.

## 17. Run the backend and test the REST integration

Starts the **action server** (custom actions, :5055) and the **Rasa REST server** (:5005) in the
background, then calls the REST endpoint from Python to confirm the integration works.

**Limitation:** Colab has no persistent public URL by default, so the cells here verify the
*backend* and the REST contract; section 18 optionally exposes a public URL for the browser UI.
Secrets are read from the environment only and are never printed; if the database is unset the
action server simply uses the JSON fallback.

In [ ]:
import subprocess, time, os, requests, json as _json

# Start the custom action server (:5055) and the Rasa REST server (:5005).
# Logs go to files (not printed) so no secrets appear in the notebook output.
env = os.environ.copy()
actions_proc = subprocess.Popen([RASA, 'run', 'actions', '--port', '5055'],
                                stdout=open('actions_server.log', 'w'), stderr=subprocess.STDOUT, env=env)
rasa_proc = subprocess.Popen([RASA, 'run', '--enable-api', '--cors', '*', '--port', '5005'],
                             stdout=open('rasa_server.log', 'w'), stderr=subprocess.STDOUT, env=env)
print('Starting action server (:5055) and Rasa REST server (:5005)...')

# Wait until the Rasa server reports ready (it loads the model on startup).
ready = False
for _ in range(90):
    try:
        if requests.get('http://localhost:5005/status', timeout=2).ok:
            ready = True; break
    except Exception:
        pass
    time.sleep(2)
print('Rasa REST server ready:', ready)

### Sample REST call

```
POST http://localhost:5005/webhooks/rest/webhook
{ "sender": "demo-user", "message": "I want to plan a trip from London to Paris" }
```

The response is a JSON array of bot messages (text plus quick-reply buttons), exactly what the
web UI consumes.

In [ ]:
URL = 'http://localhost:5005/webhooks/rest/webhook'

# 1) a free-text message
r1 = requests.post(URL, json={'sender': 'demo-user',
                              'message': 'I want to plan a trip from London to Paris'}, timeout=30)
print('HTTP', r1.status_code, '- text message response:')
print(_json.dumps(r1.json(), indent=2)[:1200])

# 2) a button payload (same format the UI sends when a chip is tapped)
r2 = requests.post(URL, json={'sender': 'demo-user',
                              'message': '/inform{"destination": "Paris"}'}, timeout=30)
print('\nHTTP', r2.status_code, '- button payload response:')
print(_json.dumps(r2.json(), indent=2)[:800])

## 18. Optional: expose the backend with a public URL

Drives the real browser UI from a local machine. Needs a free ngrok authtoken stored in Colab
**Secrets** as `NGROK_AUTHTOKEN` (never printed). Open the UI with the printed URL appended as a
query string, e.g. `frontend/index.html?rasa=<public_url>/webhooks/rest/webhook`.

On the free ngrok tier, visit the base URL once in the browser and click **Visit Site** to clear
the one-time interstitial before the UI can reach it.

In [ ]:
# Optional public tunnel for the Rasa REST server.
# Reads NGROK_AUTHTOKEN from Colab Secrets (or the environment); never printed.
token = os.environ.get('NGROK_AUTHTOKEN')
if not token and IN_COLAB:
    try:
        from google.colab import userdata
        token = userdata.get('NGROK_AUTHTOKEN')
    except Exception:
        token = None

if token:
    !pip install -q pyngrok
    from pyngrok import ngrok, conf
    conf.get_default().auth_token = token
    ngrok.kill()  # clear any previous tunnel
    public_url = ngrok.connect(5005, 'http').public_url
    print('Public Rasa REST endpoint:', public_url + '/webhooks/rest/webhook')
    print('Open the UI with  ?rasa=' + public_url + '/webhooks/rest/webhook')
else:
    print('No NGROK_AUTHTOKEN found in Colab Secrets - skipping.')
    print('This is optional: use local testing for the interactive UI instead.')

## 19. Conversation logging and analysis

After chatting in the UI, this pulls the full Rasa **tracker** for a conversation via the running
REST API and saves it to `conversation_log.json`. The tracker shows, per turn, the user text, the
predicted **intent + confidence + entities**, the **actions** taken, slot changes and the bot
replies — exactly what is needed to debug answers case by case.

Set `CONV_ID` to the sender id you want to inspect: `demo-user` for the REST demo above, or the
per-session id shown in the browser console for a UI conversation.

In [ ]:
import requests, json

CONV_ID = 'demo-user'                 # sender id to inspect (see note above)
RASA_HTTP = 'http://localhost:5005'

tr = requests.get(f'{RASA_HTTP}/conversations/{CONV_ID}/tracker', timeout=15).json()
with open('conversation_log.json', 'w') as f:
    json.dump(tr, f, indent=2, ensure_ascii=False)

print('=== Conversation:', CONV_ID, '===\n')
for e in tr.get('events', []):
    et = e.get('event')
    if et == 'user':
        it = e.get('parse_data', {}).get('intent', {})
        ents = e.get('parse_data', {}).get('entities', [])
        es = ', '.join(f"{x['entity']}={x['value']}" for x in ents) or '-'
        print(f"USER : {e.get('text','')}")
        print(f"       intent={it.get('name')} ({round(it.get('confidence',0),2)}) | entities: {es}")
    elif et == 'bot':
        print(f"BOT  : {(e.get('text') or '').replace(chr(10),' ')[:120]}")
    elif et == 'action' and e.get('name') != 'action_listen':
        print(f"ACT  : {e.get('name')}")
    elif et == 'slot':
        print(f"SLOT : {e.get('name')} = {e.get('value')}")
print('\nSaved full tracker to conversation_log.json')